# PlantPal — Week 3 Data Pipeline

Topic 3 Lab, Parts A–C. Selected pathways: **A (TXT care guides)** and **C (plant images)**.
FR8 needs source-supported care passages; FR1 needs inspectable plant images; FR11 needs source identity and version records.
This expands Week 2 C3 care documents and the C1/C2 image input. It does not run an identification or language model.

Run Cell 1 first. Connect a standard CPU Colab runtime. Upload the seven selected files through the Files panel into
/content/week3_project/data/raw: three TXT snapshots, three EXIF-stripped JPG snapshots and source_metadata.json.
Then run the remaining cells in order. No dummy ZIP, Drive mount, package installation or API key is needed for these inputs.

The nursery pages were converted from visible English web-page text to UTF-8 TXT on 2026-11-17, before upload.
Collapsed answers were not expanded. Navigation and shopping text remain for later quality review; this lab applies whitespace cleanup.
Full nursery text and derived JSONL are private because redistribution permission is unconfirmed. Notebook previews are deliberately short.
Images are CC0 photos from iNaturalist; labels come from source records, not filenames. One photo per species is only a pipeline sample.


In [2]:
# Cell 1 — Workspace setup (RUN FIRST)
from pathlib import Path
BASE_DIR = Path('/content/week3_project')
RAW_DIR = BASE_DIR / 'data' / 'raw'
PROCESSED_DIR = BASE_DIR / 'data' / 'processed'
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
RAW = RAW_DIR
PROCESSED = PROCESSED_DIR
BASE = BASE_DIR
OUT = PROCESSED_DIR
print('Raw folder:', RAW_DIR)
print('Processed folder:', PROCESSED_DIR)


Raw folder: /content/week3_project/data/raw
Processed folder: /content/week3_project/data/processed


## Part A — Upload, inspect and inventory
After the Files-panel upload, this project access step organises the explicitly named files.
It does not download or select an arbitrary file. Keep the source snapshots: /content is temporary.


In [3]:
# Access step — organise the uploaded files before Cell 3
import json
metadata_path = RAW_DIR / 'source_metadata.json'
assert metadata_path.is_file(), 'Upload the seven selected files into data/raw first.'
SOURCE_METADATA = json.loads(metadata_path.read_text(encoding='utf-8'))
for relative in SOURCE_METADATA:
    destination = RAW_DIR / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    uploaded = RAW_DIR / destination.name
    if not destination.exists() and uploaded.is_file():
        uploaded.rename(destination)
    assert destination.is_file(), f'Missing selected file: {relative}'
print('Project input paths ready.')


Project input paths ready.


In [4]:
# Cell 3 — Verify uploaded inputs
files = sorted(f for f in RAW_DIR.rglob('*') if f.is_file())
print('Input count:', len(files))
for f in files:
    print(f.relative_to(RAW_DIR), 'bytes:', f.stat().st_size)
assert files, 'Upload at least one relevant input file before proceeding.'
print('Upload verified. Select your relevant inputs for Step A3.')


Input count: 7
documents/money_plant.txt bytes: 1789
documents/snake_plant.txt bytes: 1876
documents/spider_plant.txt bytes: 1556
images/money_plant.jpg bytes: 854980
images/snake_plant.jpg bytes: 1768682
images/spider_plant.jpg bytes: 2075609
source_metadata.json bytes: 4252
Upload verified. Select your relevant inputs for Step A3.


In [5]:
# A3 — Inspect actual text and image properties
from PIL import Image
from IPython.display import display
import hashlib
for relative, details in SOURCE_METADATA.items():
    f = RAW_DIR / relative
    assert hashlib.sha256(f.read_bytes()).hexdigest() == details['sha256'], relative
    if f.suffix == '.txt':
        text = f.read_text(encoding='utf-8')
        assert text.strip() and details['species'] in text
        print(details['source_id'], relative, 'characters:', len(text), 'sample:', repr(text[:50]))
    else:
        with Image.open(f) as image:
            image.verify()
        with Image.open(f) as image:
            print(details['source_id'], relative, 'size:', image.size, 'mode:', image.mode,
                  'label:', details['source_label'], 'EXIF entries:', len(image.getexif()))
            preview = image.copy()
            preview.thumbnail((180, 180))
            display(preview)
print('A3 inspection complete: three text snapshots and three images.')


S01 documents/money_plant.txt characters: 1780 sample: 'Care guide\nEN\nBack\nMoney plant\n\nEpipremnum aureum\n'
S02 documents/snake_plant.txt characters: 1867 sample: 'Care guide\nEN\nBack\nSnake plant\n\nDracaena trifascia'
S03 documents/spider_plant.txt characters: 1549 sample: 'Care guide\nEN\nBack\nSpider plant\n\nChlorophytum como'
S04 images/money_plant.jpg size: (1536, 2048) mode: RGB label: Epipremnum aureum EXIF entries: 0
S05 images/snake_plant.jpg size: (1152, 2048) mode: RGB label: Sansevieria trifasciata EXIF entries: 0
S06 images/spider_plant.jpg size: (1536, 2048) mode: RGB label: Chlorophytum comosum EXIF entries: 0
A3 inspection complete: three text snapshots and three images.


## Part B — Prepare selected pathways and record lineage


In [6]:
# Cell 4 — Source selection and lineage utilities
from datetime import datetime, timezone
def select_source(relative_path=None, filename=None, suffixes=None):
    if relative_path:
        selected = RAW_DIR / relative_path
        if not selected.is_file():
            raise FileNotFoundError(f'Source not found: {selected}')
        return selected
    matches = sorted(p for p in RAW_DIR.rglob('*') if p.is_file()
                     and (filename is None or p.name == filename)
                     and (suffixes is None or p.suffix.lower() in suffixes))
    print('Candidates:', [str(p.relative_to(RAW_DIR)) for p in matches])
    if len(matches) != 1:
        raise ValueError('Select exactly one source using an exact relative_path.')
    return matches[0]
def source_hash(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()
def save_manifest(records):
    manifest = dict(pipeline_version='week3-v1',
                    generated_at_utc=datetime.now(timezone.utc).isoformat(), records=records)
    target = BASE_DIR / 'data' / 'week3_manifest.json'
    target.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding='utf-8')
    print('Saved lineage:', target)
    return manifest
print('Cell 4 completed successfully.')


Cell 4 completed successfully.


In [7]:
# Cell 4A — Common lineage registry
lineage_entries = {}
def register_lineage(source, output, processing):
    source, output = Path(source).resolve(), Path(output).resolve()
    if not source.is_file():
        raise FileNotFoundError(source)
    if not output.is_file() or output.stat().st_size == 0:
        raise ValueError(f'Missing or empty output: {output}')
    if not isinstance(processing, str) or not processing.strip() or processing.upper().startswith('REPLACE'):
        raise ValueError('Record the actual processing description.')
    source.relative_to(RAW_DIR.resolve())
    output.relative_to(PROCESSED_DIR.resolve())
    lineage_entries[(str(source), str(output))] = dict(source=source, output=output, processing=processing.strip())
    print('Registered:', source.name, '->', output.name)
print('Common lineage registry ready. Registered entries:', len(lineage_entries))


Common lineage registry ready. Registered entries: 0


In [20]:
# Cell 5A1 — TXT load, whitespace cleanup and chunking comparison
import re
DOC_INPUTS = [select_source(relative_path='documents/' + name + '.txt')
              for name in ['money_plant', 'snake_plant', 'spider_plant']]
def load_clean(src):
    if src.suffix.lower() != '.txt':
        raise ValueError('This selected loader expects the documented TXT snapshots.')
    text = src.read_text(encoding='utf-8')
    clean = re.sub(r'[ \t]+', ' ', text).strip()
    assert clean, f'Empty extraction: {src}'
    return text, clean
def fixed_chunks(s, size=500):
    if size <= 0:
        raise ValueError('Chunk size must be positive')
    return [s[i:i+size] for i in range(0, len(s), size)]
def paragraph_chunks(s, size=500):
    if size <= 0:
        raise ValueError('Chunk size must be positive')
    parts, current = [], ''
    for segment in re.split(r'\n\s*\n', s):
        segment = segment.strip()
        if not segment:
            continue
        if len(segment) > size:
            if current:
                parts.append(current)
                current = ''
            parts.extend(fixed_chunks(segment, size))
        elif len(current) + len(segment) + (2 if current else 0) <= size:
            current += ('\n\n' if current else '') + segment
        else:
            if current:
                parts.append(current)
            current = segment
    if current:
        parts.append(current)
    return parts
DOCUMENT_COMPARISON = []
for f in DOC_INPUTS:
    text, clean = load_clean(f)
    fixed, structured = fixed_chunks(clean, 500), paragraph_chunks(clean, 500)
    result = dict(source=f.name, raw_characters=len(text), cleaned_characters=len(clean),
                  fixed_chunks=len(fixed), paragraph_chunks=len(structured),
                  fixed_lengths=[len(x) for x in fixed], paragraph_lengths=[len(x) for x in structured])
    DOCUMENT_COMPARISON.append(result)
    print(json.dumps(result))
    print('Fixed first chunk end:', repr(fixed[0][-18:]))
    print('Paragraph first chunk end:', repr(structured[0][-18:]))
print('Choose using the counts and boundary samples; paragraph-aware chunks are not semantic chunks.')


{"source": "money_plant.txt", "raw_characters": 1780, "cleaned_characters": 1780, "fixed_chunks": 4, "paragraph_chunks": 5, "fixed_lengths": [500, 500, 500, 280], "paragraph_lengths": [369, 496, 404, 466, 37]}
Fixed first chunk end: 'sun.\nWhat does thi'
Paragraph first chunk end: 'hours of a drink.”'
{"source": "snake_plant.txt", "raw_characters": 1867, "cleaned_characters": 1867, "fixed_chunks": 4, "paragraph_chunks": 6, "fixed_lengths": [500, 500, 500, 367], "paragraph_lengths": [390, 500, 56, 410, 466, 37]}
Fixed first chunk end: 'ht shade.\nWhat doe'
Paragraph first chunk end: 'han from neglect.”'
{"source": "spider_plant.txt", "raw_characters": 1549, "cleaned_characters": 1549, "fixed_chunks": 4, "paragraph_chunks": 5, "fixed_lengths": [500, 500, 500, 49], "paragraph_lengths": [316, 370, 352, 466, 37]}
Fixed first chunk end: 'pot\nOrdinary potti'
Paragraph first chunk end: ' make new plants.”'
Choose using the counts and boundary samples; paragraph-aware chunks are not semantic c

In [9]:
# Cell 5A2 — Selected Part B document export, reused in Part C
CHOSEN_METHOD = 'paragraph'
CONFIG_A = {'chunk_method': CHOSEN_METHOD, 'chunk_size': 500}
def text_records(inputs, cfg):
    records = []
    splitter = paragraph_chunks if cfg['chunk_method'] == 'paragraph' else fixed_chunks
    for src in inputs:
        _, clean = load_clean(src)
        for i, chunk in enumerate(splitter(clean, cfg['chunk_size'])):
            records.append(dict(record_id=f'{src.stem}-{i:04d}', text=chunk,
                source=str(src.relative_to(RAW_DIR)), source_sha256=source_hash(src),
                processing=f"{cfg['chunk_method']}-aware-char-{cfg['chunk_size']}", page=None))
    return records
prepared_records = text_records(DOC_INPUTS, CONFIG_A)
prepared_path = PROCESSED_DIR / 'prepared_text.jsonl'
prepared_path.write_text(''.join(json.dumps(r, ensure_ascii=False) + '\n' for r in prepared_records), encoding='utf-8')
print('Selected:', CONFIG_A, 'records:', len(prepared_records), 'output:', prepared_path)
print('Accepted limitation: long paragraphs can still split; navigation and advice accuracy need Topic 4 review.')


Selected: {'chunk_method': 'paragraph', 'chunk_size': 500} records: 16 output: /content/week3_project/data/processed/prepared_text.jsonl
Accepted limitation: long paragraphs can still split; navigation and advice accuracy need Topic 4 review.


In [10]:
# B3 — Validate text output; B4.1 — register each contributing source
assert prepared_path.is_file() and prepared_path.stat().st_size > 0
reloaded = [json.loads(line) for line in prepared_path.read_text(encoding='utf-8').splitlines()]
assert len(reloaded) == len(prepared_records)
assert len({r['record_id'] for r in reloaded}) == len(reloaded)
assert all(r['text'].strip() and len(r['text']) <= CONFIG_A['chunk_size'] for r in reloaded)
for src in DOC_INPUTS:
    subset = [r for r in reloaded if r['source'] == str(src.relative_to(RAW_DIR))]
    assert subset and all(r['source_sha256'] == source_hash(src) and r['page'] is None for r in subset)
    print('Reloaded sample:', subset[0]['record_id'], 'length:', len(subset[0]['text']), 'page:', subset[0]['page'])
    register_lineage(src, prepared_path, 'TXT read; whitespace cleanup; paragraph-aware-char-500; no page numbers')
print('B3 documents: PASS; records:', len(reloaded))


Reloaded sample: money_plant-0000 length: 369 page: None
Registered: money_plant.txt -> prepared_text.jsonl
Reloaded sample: snake_plant-0000 length: 390 page: None
Registered: snake_plant.txt -> prepared_text.jsonl
Reloaded sample: spider_plant-0000 length: 316 page: None
Registered: spider_plant.txt -> prepared_text.jsonl
B3 documents: PASS; records: 16


In [11]:
# Cell 5C — Image inspection, two settings and selected export
import csv
IMAGE_INPUTS = [select_source(relative_path='images/' + name + '.jpg')
                for name in ['money_plant', 'snake_plant', 'spider_plant']]
CONFIG_C = {'resize': None, 'convert_rgb': True, 'strip_metadata': True}
def prepared_image(src, cfg):
    with Image.open(src) as im:
        image = im.convert('RGB') if cfg['convert_rgb'] else im.copy()
        if cfg['resize'] is not None:
            image = image.resize(tuple(cfg['resize']))
        return Image.frombytes(image.mode, image.size, image.tobytes())
IMAGE_COMPARISON, image_rows = [], []
image_out = PROCESSED_DIR / 'images'
image_out.mkdir(exist_ok=True)
for i, src in enumerate(IMAGE_INPUTS, 1):
    with Image.open(src) as im:
        im.verify()
    with Image.open(src) as im:
        original_size, original_mode = list(im.size), im.mode
    square = prepared_image(src, {'resize': [224, 224], 'convert_rgb': True})
    selected = prepared_image(src, CONFIG_C)
    comparison = dict(source=src.name, original_size=original_size, original_mode=original_mode,
                      square_size=list(square.size), selected_size=list(selected.size),
                      original_aspect=round(original_size[0]/original_size[1], 4), square_aspect=1.0)
    IMAGE_COMPARISON.append(comparison)
    print(json.dumps(comparison))
    if i == 1:
        print('224-square comparison preview:')
        display(square)
        preview = selected.copy()
        preview.thumbnail((224, 224))
        print('Selected aspect-preserving preview:')
        display(preview)
    output = image_out / f'prepared_{i:03d}.png'
    selected.save(output)
    details = SOURCE_METADATA[str(src.relative_to(RAW_DIR))]
    image_rows.append(dict(source=str(src.relative_to(RAW_DIR)), source_sha256=source_hash(src),
        output=str(output.relative_to(BASE_DIR)), original_width=original_size[0], original_height=original_size[1],
        original_mode=original_mode, width=selected.width, height=selected.height, mode=selected.mode,
        species=details['species'], source_label=details['source_label'], owner=details['owner'],
        source_url=details['url'], photo_id=details['photo_id'], licence='CC0-1.0',
        licence_url='https://creativecommons.org/publicdomain/zero/1.0/',
        processing='RGB; resize=None; strip_metadata=True'))
manifest_csv = PROCESSED_DIR / 'image_manifest.csv'
with manifest_csv.open('w', newline='', encoding='utf-8') as handle:
    writer = csv.DictWriter(handle, fieldnames=list(image_rows[0]))
    writer.writeheader()
    writer.writerows(image_rows)
print('Selected:', CONFIG_C, 'PNG files:', len(image_rows), 'manifest:', manifest_csv)
print('Accepted limitation: larger PNGs; no identification service input size has been selected yet.')


{"source": "money_plant.jpg", "original_size": [1536, 2048], "original_mode": "RGB", "square_size": [224, 224], "selected_size": [1536, 2048], "original_aspect": 0.75, "square_aspect": 1.0}
224-square comparison preview:
Selected aspect-preserving preview:
{"source": "snake_plant.jpg", "original_size": [1152, 2048], "original_mode": "RGB", "square_size": [224, 224], "selected_size": [1152, 2048], "original_aspect": 0.5625, "square_aspect": 1.0}
{"source": "spider_plant.jpg", "original_size": [1536, 2048], "original_mode": "RGB", "square_size": [224, 224], "selected_size": [1536, 2048], "original_aspect": 0.75, "square_aspect": 1.0}
Selected: {'resize': None, 'convert_rgb': True, 'strip_metadata': True} PNG files: 3 manifest: /content/week3_project/data/processed/image_manifest.csv
Accepted limitation: larger PNGs; no identification service input size has been selected yet.


In [12]:
# B3 — Reload images and CSV; B4.1 — register both output relationships
with manifest_csv.open(encoding='utf-8', newline='') as handle:
    saved_rows = list(csv.DictReader(handle))
assert len(saved_rows) == len(IMAGE_INPUTS) == 3
for src, row in zip(IMAGE_INPUTS, saved_rows):
    output = BASE_DIR / row['output']
    assert output.is_file() and output.stat().st_size > 0
    with Image.open(output) as image:
        image.verify()
    with Image.open(output) as image:
        assert image.mode == 'RGB' and list(image.size) == [int(row['width']), int(row['height'])]
        assert not image.getexif() and not image.info
        print('Reloaded:', output.name, image.size, image.mode, 'metadata:', image.info)
    register_lineage(src, output, row['processing'])
    register_lineage(src, manifest_csv, 'Source-provided label and attribution; dimensions and processing metadata')
assert manifest_csv.stat().st_size > 0
print('B3 images: PASS; image records:', len(saved_rows))


Reloaded: prepared_001.png (1536, 2048) RGB metadata: {}
Registered: money_plant.jpg -> prepared_001.png
Registered: money_plant.jpg -> image_manifest.csv
Reloaded: prepared_002.png (1152, 2048) RGB metadata: {}
Registered: snake_plant.jpg -> prepared_002.png
Registered: snake_plant.jpg -> image_manifest.csv
Reloaded: prepared_003.png (1536, 2048) RGB metadata: {}
Registered: spider_plant.jpg -> prepared_003.png
Registered: spider_plant.jpg -> image_manifest.csv
B3 images: PASS; image records: 3


In [13]:
# Cell 6A — Generate common lineage manifest
assert lineage_entries, 'Register each selected pathway first.'
lineage_records = []
for entry in lineage_entries.values():
    source, output = entry['source'], entry['output']
    relative = str(source.relative_to(RAW_DIR))
    info = SOURCE_METADATA[relative]
    assert source.is_file() and output.is_file() and output.stat().st_size > 0
    lineage_records.append(dict(source_id=info['source_id'], source_filename=relative,
        source_sha256=source_hash(source), prepared_output=str(output.relative_to(BASE_DIR)),
        prepared_sha256=source_hash(output), processing_note=entry['processing'],
        source_version=info['version'], access_rights=info['access_rights'], source_url=info['url'],
        status='Prepared; quality review pending'))
manifest = save_manifest(lineage_records)
print('COMMON LINEAGE MANIFEST GENERATED. Total lineage entries:', len(manifest['records']))


Saved lineage: /content/week3_project/data/week3_manifest.json
COMMON LINEAGE MANIFEST GENERATED. Total lineage entries: 9


In [14]:
# Cell 6B — Verify common lineage manifest
manifest_path = BASE_DIR / 'data' / 'week3_manifest.json'
saved_manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
assert saved_manifest['records'] and len(saved_manifest['records']) == len(lineage_entries)
for record in saved_manifest['records']:
    source, output = RAW_DIR / record['source_filename'], BASE_DIR / record['prepared_output']
    assert source.is_file() and output.is_file()
    assert source_hash(source) == record['source_sha256']
    assert source_hash(output) == record['prepared_sha256']
print('COMMON LINEAGE VALIDATION: PASS')
print('Verified records:', len(saved_manifest['records']))


COMMON LINEAGE VALIDATION: PASS
Verified records: 9


## Part C — Executable V1 pipelines
The common lab utility is adapted to accept the folder of selected TXT snapshots and carry known source rights.
Parts B and C reuse the same preparation functions and configurations.


In [15]:
# Cell 7 — Common V1 pipeline utilities
PIPE_DIR = BASE_DIR / 'data' / 'pipeline_v1'
PIPE_DIR.mkdir(parents=True, exist_ok=True)
PIPELINE_RUNS = {}
def sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()
def resolve_source(relative_path, folder=False):
    source = RAW_DIR / relative_path
    if not (source.is_dir() if folder else source.is_file()):
        raise FileNotFoundError(source)
    source.resolve().relative_to(RAW_DIR.resolve())
    return source
def execute_pipeline(kind, source, inspect_fn, prepare_fn, configuration):
    source = Path(source)
    suffixes = {'.txt'} if kind == 'documents' else {'.png', '.jpg', '.jpeg', '.webp'}
    inputs = sorted(p for p in source.iterdir() if p.is_file() and p.suffix.lower() in suffixes)
    if not inputs or any(not p.is_file() for p in inputs):
        raise ValueError('No valid source files to ingest.')
    for p in inputs:
        details = SOURCE_METADATA[str(p.relative_to(RAW_DIR))]
        assert details['access_rights'] and sha256_file(p) == details['sha256']
    inspection = inspect_fn(inputs, configuration)
    output_dir = PIPE_DIR / kind
    output_dir.mkdir(parents=True, exist_ok=True)
    outputs, mappings, method = prepare_fn(inputs, output_dir, configuration)
    outputs = [Path(p) for p in outputs]
    if not outputs:
        raise ValueError('No prepared outputs were produced.')
    for output in outputs:
        if not output.is_file() or output.stat().st_size == 0:
            raise ValueError(f'Missing or empty output: {output}')
    assert {m['source'] for m in mappings} == {str(p) for p in inputs}
    assert {m['output'] for m in mappings} == {str(p) for p in outputs}
    validation = dict(output_count=len(outputs), output_sha256=[sha256_file(p) for p in outputs],
                      output_bytes=[p.stat().st_size for p in outputs])
    record = dict(pipeline_version='week3-v1', generated_at_utc=datetime.now(timezone.utc).isoformat(),
        data_type=kind, configuration=configuration, processing_method=method,
        sources=[dict(path=str(p), sha256=sha256_file(p)) for p in inputs],
        outputs=[str(p) for p in outputs], source_output_mapping=mappings,
        inspection=inspection, validation=validation,
        source_version={str(p.relative_to(RAW_DIR)): SOURCE_METADATA[str(p.relative_to(RAW_DIR))]['version'] for p in inputs},
        access_rights={str(p.relative_to(RAW_DIR)): SOURCE_METADATA[str(p.relative_to(RAW_DIR))]['access_rights'] for p in inputs})
    manifest_path = output_dir / 'pipeline_manifest.json'
    manifest_path.write_text(json.dumps(record, indent=2, ensure_ascii=False), encoding='utf-8')
    print(f'SUCCESS: {kind} | Inputs: {len(inputs)} | Outputs: {len(outputs)}')
    print('Manifest:', manifest_path)
    return record
def register_and_run(kind, source, inspect_fn, prepare_fn, configuration):
    def runner():
        return execute_pipeline(kind, source, inspect_fn, prepare_fn, configuration)
    first = runner()
    PIPELINE_RUNS[kind] = {'first': first, 'runner': runner}
    return first
print('Cell 7 ready. No source processed at this stage.')


Cell 7 ready. No source processed at this stage.


In [16]:
# Cell 8A — Document pipeline with the Part B decision
SOURCE_A = resolve_source('documents', folder=True)
def inspect_documents(inputs, cfg):
    samples = []
    for src in inputs:
        raw, clean = load_clean(src)
        samples.append(dict(name=src.name, raw_characters=len(raw), cleaned_characters=len(clean)))
    return dict(document_count=len(inputs), sample=samples)
def prepare_documents(inputs, out, cfg):
    records = text_records(inputs, cfg)
    target = out / 'prepared_text.jsonl'
    target.write_text(''.join(json.dumps(r, ensure_ascii=False) + '\n' for r in records), encoding='utf-8')
    reloaded = [json.loads(line) for line in target.read_text(encoding='utf-8').splitlines()]
    assert len(reloaded) == len(records) and all(r['text'].strip() for r in reloaded)
    assert len({r['record_id'] for r in reloaded}) == len(reloaded)
    print('Prepared record inspected:', reloaded[0]['record_id'], 'length:', len(reloaded[0]['text']))
    mappings = [dict(source=str(src), output=str(target)) for src in inputs]
    return [target], mappings, 'TXT read; whitespace cleanup; paragraph-aware-char-500; page=None'
run_A = register_and_run('documents', SOURCE_A, inspect_documents, prepare_documents, CONFIG_A)
assert sha256_file(PIPE_DIR / 'documents' / 'prepared_text.jsonl') == sha256_file(PROCESSED_DIR / 'prepared_text.jsonl')
print('Part B/Part C document output equality: PASS')


Prepared record inspected: money_plant-0000 length: 369
SUCCESS: documents | Inputs: 3 | Outputs: 1
Manifest: /content/week3_project/data/pipeline_v1/documents/pipeline_manifest.json
Part B/Part C document output equality: PASS


In [17]:
# Cell 8C — Image pipeline with the Part B decision
SOURCE_C = resolve_source('images', folder=True)
def inspect_images(inputs, cfg):
    properties = []
    for src in inputs:
        with Image.open(src) as im:
            im.verify()
        with Image.open(src) as im:
            properties.append(dict(name=src.name, size=list(im.size), mode=im.mode))
    return dict(image_count=len(inputs), sample=properties[:3])
def prepare_images(inputs, out, cfg):
    outputs, mappings = [], []
    for i, src in enumerate(inputs, 1):
        image = prepared_image(src, cfg)
        target = out / f'prepared_{i:03d}.png'
        image.save(target)
        with Image.open(target) as saved:
            saved.verify()
        with Image.open(target) as saved:
            assert saved.mode == 'RGB' and saved.size == image.size and not saved.getexif() and not saved.info
        outputs.append(target)
        mappings.append(dict(source=str(src), output=str(target)))
    return outputs, mappings, 'convert_rgb=True; resize=None; strip_metadata=True'
run_C = register_and_run('images', SOURCE_C, inspect_images, prepare_images, CONFIG_C)
for path in run_C['outputs']:
    assert sha256_file(path) == sha256_file(PROCESSED_DIR / 'images' / Path(path).name)
print('Part B/Part C image output equality: PASS')


SUCCESS: images | Inputs: 3 | Outputs: 3
Manifest: /content/week3_project/data/pipeline_v1/images/pipeline_manifest.json
Part B/Part C image output equality: PASS


In [18]:
# Cell 9 — Required repeatability and provenance verification
assert PIPELINE_RUNS, 'Run a selected Cell 8 pathway first.'
print('--- Repeatability Verification ---')
for kind, entry in PIPELINE_RUNS.items():
    first, second = entry['first'], entry['runner']()
    assert first['sources'] == second['sources'], f'Source fingerprint changed for {kind}'
    assert first['validation'] == second['validation'], f'Prepared output changed for {kind}'
    assert first['source_output_mapping'] == second['source_output_mapping'], f'Mapping changed for {kind}'
    print(f'PASS: {kind} — source fingerprints, prepared outputs and mappings match')
print('All selected pipelines passed repeatability smoke checks.')
print('These checks do not establish model accuracy, care safety or cross-environment reproducibility.')


--- Repeatability Verification ---
Prepared record inspected: money_plant-0000 length: 369
SUCCESS: documents | Inputs: 3 | Outputs: 1
Manifest: /content/week3_project/data/pipeline_v1/documents/pipeline_manifest.json
PASS: documents — source fingerprints, prepared outputs and mappings match
SUCCESS: images | Inputs: 3 | Outputs: 3
Manifest: /content/week3_project/data/pipeline_v1/images/pipeline_manifest.json
PASS: images — source fingerprints, prepared outputs and mappings match
All selected pipelines passed repeatability smoke checks.
These checks do not establish model accuracy, care safety or cross-environment reproducibility.


## C4 — Preserve the executed artefacts
Download this executed notebook using File → Download → Download .ipynb.
The transfer cell packages the lab outputs before the temporary runtime is lost.
Keep nursery snapshots and derived text private; commit the CC0 images, manifests, notebook and non-sensitive evidence.


In [19]:
# Transfer the required lab outputs out of temporary /content storage
import shutil
from google.colab import files as colab_files
archive = shutil.make_archive('/content/plantpal_week3_outputs', 'zip', BASE_DIR)
print('Preserve this ZIP privately: it contains full care text. Do not upload the ZIP to public GitHub.')
colab_files.download(archive)


Preserve this ZIP privately: it contains full care text. Do not upload the ZIP to public GitHub.
